<a href="https://colab.research.google.com/github/hamnasz/Urdu-OCR-Project-Code-Saviours-SI-26-Humna-Imran/blob/main/SI26_Week1_Humna_Imran.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Step 6, Source 1 — Download UTRSet-Real directly in Colab

The Hugging Face dataset card for abdur75648/UTRSet-Real is just a
placeholder (no actual data files) -- confirmed by checking the repo
directly: it says "currently empty", 4.21 kB total. The real data lives
on Google Drive, linked from the official GitHub repo for the UTRNet
paper (ICDAR 2023):
  https://github.com/abdur75648/UTRNet-High-Resolution-Urdu-Text-Recognition

This script downloads it via gdown (works fine from Colab, since Colab
has its own internet access), unzips it, and converts whatever ground-
truth format it ships in (commonly `gt.txt` with `path\tlabel` per line,
per the dataset's own repo convention) into your project's
data/labels.csv format, plus copies a sample of images into
data/raw/other/.

Run in Colab:
    !pip install gdown -q
    !python download_utrset_real.py
"""

import csv
import os
import shutil
import subprocess
import zipfile

# Google Drive file ID extracted from:
# https://drive.google.com/file/d/1mABkzaWe1hLikXCaM5nmLsBCWtxvDE7T/view
GDRIVE_FILE_ID = "1mABkzaWe1hLikXCaM5nmLsBCWtxvDE7T"

DOWNLOAD_DIR = "utrset_real_download"
ZIP_PATH = os.path.join(DOWNLOAD_DIR, "utrset_real.zip")
EXTRACT_DIR = os.path.join(DOWNLOAD_DIR, "extracted")

OUT_DIR = "data/raw/other"
LABELS_CSV = "data/labels.csv"
N_SAMPLES = 60  # how many of the 11,000+ lines to actually use for Week 1


def download():
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    if os.path.exists(ZIP_PATH):
        print(f"Already downloaded: {ZIP_PATH}")
        return

    print("Downloading UTRSet-Real from Google Drive (this is ~hundreds of MB, may take a few minutes)...")
    try:
        import gdown
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)
    except ImportError:
        # fallback to gdown CLI if the python import path is unavailable
        subprocess.run(
            ["gdown", "--id", GDRIVE_FILE_ID, "-O", ZIP_PATH],
            check=True,
        )

    if not os.path.exists(ZIP_PATH) or os.path.getsize(ZIP_PATH) == 0:
        raise RuntimeError(
            "Download failed or produced an empty file. Google Drive "
            "sometimes blocks automated downloads of large files with a "
            "'too many downloads' warning page instead of the real file. "
            "If this happens: open the link in your own browser "
            "(https://drive.google.com/file/d/1mABkzaWe1hLikXCaM5nmLsBCWtxvDE7T/view), "
            "click through the warning, download manually, then upload "
            "the zip to Colab and re-run this script -- it'll detect the "
            "existing zip and skip straight to extraction."
        )
    print(f"Downloaded: {ZIP_PATH} ({os.path.getsize(ZIP_PATH) / 1e6:.1f} MB)")


def extract():
    if os.path.exists(EXTRACT_DIR) and os.listdir(EXTRACT_DIR):
        print(f"Already extracted: {EXTRACT_DIR}")
        return
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print("Extracted.")


def find_gt_file(root):
    """Locate the ground-truth label file -- commonly gt.txt per the
    dataset author's own repo convention, but search broadly in case
    the zip structure differs."""
    candidates = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if fname.lower() in ("gt.txt", "ground_truth.txt", "labels.txt"):
                candidates.append(os.path.join(dirpath, fname))
    return candidates


def find_images(root):
    images = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                images.append(os.path.join(dirpath, fname))
    return images


def main():
    download()
    extract()

    gt_files = find_gt_file(EXTRACT_DIR)
    images = find_images(EXTRACT_DIR)

    print(f"\nFound {len(gt_files)} ground-truth file(s), {len(images)} image(s) in the extracted archive.")

    os.makedirs(OUT_DIR, exist_ok=True)

    new_rows = []

    if gt_files:
        # Parse the first gt.txt-style file found.
        gt_path = gt_files[0]
        print(f"Parsing ground truth from: {gt_path}")
        with open(gt_path, "r", encoding="utf-8") as f:
            lines = [l.strip() for l in f if l.strip()]

        gt_dir = os.path.dirname(gt_path)
        count = 0
        for line in lines:
            if count >= N_SAMPLES:
                break
            # Expected format: relative/image/path.png\tlabel text
            if "\t" in line:
                img_rel_path, text = line.split("\t", 1)
            elif " " in line:
                # fallback: some gt files use a single space, not tab
                img_rel_path, text = line.split(" ", 1)
            else:
                continue

            src_img_path = os.path.join(gt_dir, img_rel_path)
            if not os.path.exists(src_img_path):
                # try resolving relative to extract root instead
                src_img_path = os.path.join(EXTRACT_DIR, img_rel_path)
            if not os.path.exists(src_img_path):
                continue

            fname = f"utrset_{count:03d}.png"
            dst_path = os.path.join(OUT_DIR, fname)
            shutil.copy(src_img_path, dst_path)
            new_rows.append({"image": dst_path, "text": text})
            count += 1

        print(f"Copied {count} images with verified ground-truth labels.")

    elif images:
        # No gt.txt found -- unexpected, but handle gracefully: copy
        # images anyway and flag them for manual labeling rather than
        # silently producing nothing.
        print("WARNING: No ground-truth file found automatically.")
        print("Copying a sample of images anyway -- you'll need to find")
        print("the label source manually (check the extracted folder")
        print(f"structure at {EXTRACT_DIR}) or label these by eye.")
        for i, img_path in enumerate(images[:N_SAMPLES]):
            fname = f"utrset_{i:03d}.png"
            dst_path = os.path.join(OUT_DIR, fname)
            shutil.copy(img_path, dst_path)
            new_rows.append({"image": dst_path, "text": "FILL_IN_URDU_TEXT_HERE"})
    else:
        raise RuntimeError(
            f"No images or ground-truth files found under {EXTRACT_DIR}. "
            f"Check the actual folder structure with: "
            f"!find {EXTRACT_DIR} -maxdepth 3"
        )

    # Append to existing labels.csv (don't overwrite synthetic-data rows)
    existing_rows = []
    if os.path.exists(LABELS_CSV):
        with open(LABELS_CSV, "r", encoding="utf-8") as f:
            existing_rows = list(csv.DictReader(f))
        print(f"\nFound existing labels.csv with {len(existing_rows)} rows -- appending.")
    else:
        os.makedirs(os.path.dirname(LABELS_CSV), exist_ok=True)

    all_rows = existing_rows + new_rows
    with open(LABELS_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["image", "text"])
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"labels.csv now has {len(all_rows)} total entries.")
    print(f"\nDataset license: CC BY-NC-SA 4.0 (non-commercial, research use only).")
    print("Cite in your README:")
    print("""
  Rahman, A., Ghosh, A., Arora, C. (2023). UTRNet: High-Resolution Urdu
  Text Recognition in Printed Documents. In: Document Analysis and
  Recognition - ICDAR 2023. Springer Nature Switzerland.
""")


if __name__ == "__main__":
    main()

Downloading...
From (original): https://drive.google.com/uc?id=1mABkzaWe1hLikXCaM5nmLsBCWtxvDE7T
From (redirected): https://drive.google.com/uc?id=1mABkzaWe1hLikXCaM5nmLsBCWtxvDE7T&confirm=t&uuid=2427de5d-ab03-4e20-ae04-798f73be833b
To: /content/utrset_real_download/utrset_real.zip
100%|██████████| 212M/212M [00:03<00:00, 64.6MB/s]


Downloaded: utrset_real_download/utrset_real.zip (211.7 MB)
Extracting...
Extracted.

Found 2 ground-truth file(s), 22322 image(s) in the extracted archive.
Parsing ground truth from: utrset_real_download/extracted/UTRSet-Real/train/gt.txt
Copied 60 images with verified ground-truth labels.
labels.csv now has 60 total entries.

Dataset license: CC BY-NC-SA 4.0 (non-commercial, research use only).
Cite in your README:

  Rahman, A., Ghosh, A., Arora, C. (2023). UTRNet: High-Resolution Urdu
  Text Recognition in Printed Documents. In: Document Analysis and
  Recognition - ICDAR 2023. Springer Nature Switzerland.

